#Basic structure
####First the web is scrapped for news articles where the text and titles are collected
Next, The related pdf created is submitted to Page index in order to create a tree structure

Then, three models are tested

One is pure PageIndex where the llm is called twice once to assess the relevant nodes and other to use the context provided,

The second is page index tree but the nodes are used for  bm25 retrieval to provide the relevant nodes and finally llm call to use the context

Last one is Naive where there is direct llm call and all the complete corpus is provided .


In [1]:
import requests
from bs4 import BeautifulSoup
import nltk
import re
!pip install langchain_core
from google.colab import userdata
import os
import json
import asyncio
!pip install pageindex
!pip install reportlab
!pip install langchain
!pip install langchain-google-genai
!pip install google-genai
!pip install pypdf
from langchain_core.documents import Document
from pageindex import PageIndexClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from google import genai
import pageindex.utils as utils
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib.styles import getSampleStyleSheet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 7.3 MB/s eta 0:00:00


In [2]:
urls=[]
urls.append("https://timesofindia.indiatimes.com/world/middle-east/enriched-uranium-must-remain-inside-iran-mojtaba-khamenei-rejects-key-us-demand-amid-fragile-ceasefire-talks/articleshow/131245755.cms")
urls.append("https://timesofindia.indiatimes.com/india/i-vadasseri-damodara-menon-satheesan-why-kerala-congress-leaders-are-not-happy-with-their-own-cm/articleshow/131247649.cms")
urls.append("https://timesofindia.indiatimes.com/india/pm-modi-meloni-chemistry-how-italy-became-the-heart-of-indias-europe-bet/articleshow/131248103.cms")
urls.append("https://timesofindia.indiatimes.com/business/india-business/commercial-lpg-gets-costlier-cylinder-price-raised-by-rs-42-53-50-to-now-cost-rs-3113-5-in-delhi/articleshow/131431856.cms")
urls.append("https://timesofindia.indiatimes.com/city/bengaluru/dks-sidda-head-to-delhi-to-finalise-new-karnataka-team/articleshow/131431973.cms")

urls.append("https://timesofindia.indiatimes.com/technology/tech-news/days-after-michael-and-susan-dells-more-than-6-billion-donation-trump-accounts-app-goes-live/articleshow/131429029.cms")
urls.append("https://timesofindia.indiatimes.com/world/europe/multimillion-dollar-banana-french-museum-artwork-comedian-eaten-twice-before-is-now-stolen/articleshow/131432076.cms")
urls.append("https://timesofindia.indiatimes.com/india/the-big-hapus-crisis-how-climate-shocks-are-threatening-indias-iconic-alphonso-mango/articleshow/131422923.cms")
urls.append("https://timesofindia.indiatimes.com/india/sc-law-is-against-commercialising-prostitution-doesnt-want-to-ban-it/articleshow/131431443.cms")
urls.append("https://timesofindia.indiatimes.com/world/middle-east/iran-president-pezeshkian-submits-resignation-letter-to-supreme-leaders-office-claims-report-official-rejects-claim/articleshow/131431370.cms")

urls.append("https://timesofindia.indiatimes.com/world/rest-of-world/viral-post-claims-indian-woman-caught-shoplifting-in-japan-tried-to-bribe-shopkeeper-police-they-respect-india-a-lot-/articleshow/131426270.cms")
urls.append("https://timesofindia.indiatimes.com/health/why-more-young-adults-are-developing-heart-disease-the-hidden-impact-of-sitting-stress-and-urban-pollution/photostory/131423754.cms")
urls.append("https://timesofindia.indiatimes.com/world/us/low-rating-disaster-trump-blasts-cnn-for-claims-that-iran-deal-does-not-cover-nuclear/articleshow/131431916.cms")
urls.append("https://timesofindia.indiatimes.com/city/hubballi/state-approves-30000-guest-teachers-to-tackle-school-staff-shortage/articleshow/131409010.cms")
urls.append("https://timesofindia.indiatimes.com/city/bengaluru/cabinet-age-debate-puts-veterans-on-edge/articleshow/131413833.cms")

urls.append("https://timesofindia.indiatimes.com/science/green-breakthrough-mouse-eyes-performed-photosynthesis-in-a-remarkable-experiment-that-could-transform-eye-care/articleshow/131435294.cms")
urls.append("https://timesofindia.indiatimes.com/defence/international/two-wars-one-arsenal-ukraine-or-israel-who-gets-americas-missiles-when-supplies-run-short/articleshow/131263370.cms")
urls.append("https://timesofindia.indiatimes.com/business/india-business/indias-tryst-with-oil-shocks-how-crude-prices-shaped-economy-and-policy-in-last-five-decades/articleshow/131265685.cms")
urls.append("https://timesofindia.indiatimes.com/world/middle-east/warning-shot-from-iran-uae-nuclear-plant-attack-from-iraq-sparks-fears-of-wider-gulf-war-as-tehrans-proxies-step-in/articleshow/131264706.cms")

print(len(urls))
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

19


In [4]:
import time

In [3]:
import inflect
#Just some preprocessing functions

def num2words(text):
    p = inflect.engine()


    temp = text.split()
    s2 = []
    for word in temp:
        if word.isdigit():
            s2.append(p.number_to_words(word))
        else:
            s2.append(word)
    res = ' '.join(s2)
    return res

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('punkt')

def title_extracter(string):
    list1=string.split("/")
    for sections in list1:
        list2=sections.split("-")
        if len(list2)>3:
            return " ".join(list2)
title_extracter(urls[0])


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


'enriched uranium must remain inside iran mojtaba khamenei rejects key us demand amid fragile ceasefire talks'

In [9]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
PAGEINDEX_API_KEY = userdata.get('First_API_PAGE_INDEX_Key_alt')
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)


In [10]:
# As we know page index uses the pdf structure to help it split the corpus up into semantically related chunks,
# hence each article is provided in a new page to achieve so
#And as collected articles had no paragraph structure i artificially added by combining a few lines
def submit_pdf(articles):
    story=[]
    pdf_file = "/content/output.pdf"
    doc = SimpleDocTemplate(pdf_file, pagesize=letter)
    styles = getSampleStyleSheet()
    chunk_size = 4

    for article in articles:

        story.append(
            Paragraph(
                article["title"],
                styles["Heading1"]
            )
        )

        story.append(Spacer(1, 12))


        paragraphs = []
        sentences = re.split(r'(?<=[.!?])\s+', article["text"])


        for i in range(0, len(sentences), chunk_size):

            para = " ".join(sentences[i:i+chunk_size])
            paragraphs.append(para)
        print(len(paragraphs))

        for para in paragraphs:

            if para.strip():

                story.append(
                    Paragraph(
                        para,
                        styles["BodyText"]
                    )
                )

                story.append(Spacer(1, 8))

        story.append(PageBreak())
    doc.build(story)
    result = pi_client.submit_document(pdf_file)
    return(result["doc_id"])

In [41]:
#Here all data from the 19 articles is collected and stored into the tree structure,
# as well as stored normally for the Naive model later
#The numbers are no. of paragraphs per article
content=[]
complete_content_list=[]
for url in urls:
  response = requests.get(url, headers=headers)
  response.encoding = 'utf-8'
  soup = BeautifulSoup(response.text, 'html.parser')
  text_req = soup.find('div', {"data-articlebody": "1"}).text
  print(url)
  title = title_extracter(url)
  text_req = num2words(text_req)
  text_req = re.sub(r'\s+', ' ', text_req)
  text_req = text_req.lower()
  dict1={"title":title,"text":text_req}
  content.append(dict1)
  complete_content_list.append("".join([title,text_req]))
doc_id = submit_pdf(content)
complete_context="".join(complete_content_list)




https://timesofindia.indiatimes.com/world/middle-east/enriched-uranium-must-remain-inside-iran-mojtaba-khamenei-rejects-key-us-demand-amid-fragile-ceasefire-talks/articleshow/131245755.cms
https://timesofindia.indiatimes.com/india/i-vadasseri-damodara-menon-satheesan-why-kerala-congress-leaders-are-not-happy-with-their-own-cm/articleshow/131247649.cms
https://timesofindia.indiatimes.com/india/pm-modi-meloni-chemistry-how-italy-became-the-heart-of-indias-europe-bet/articleshow/131248103.cms
https://timesofindia.indiatimes.com/business/india-business/commercial-lpg-gets-costlier-cylinder-price-raised-by-rs-42-53-50-to-now-cost-rs-3113-5-in-delhi/articleshow/131431856.cms
https://timesofindia.indiatimes.com/city/bengaluru/dks-sidda-head-to-delhi-to-finalise-new-karnataka-team/articleshow/131431973.cms
https://timesofindia.indiatimes.com/technology/tech-news/days-after-michael-and-susan-dells-more-than-6-billion-donation-trump-accounts-app-goes-live/articleshow/131429029.cms
https://timeso

In [ ]:
#This is not relevant just needed for seeing what preprocessing techniques were needed
file_path = "/content/drive/MyDrive/Scrapped Articles/Article1_unprocessed.txt"

content1 = "\n\n".join(doc.page_content for doc in content)

with open(file_path, "w", encoding="utf-8") as f:
    f.write(content1)



In [ ]:
print(doc_id)

pi-cmpw2m54l024o01qurgrp3mq9


In [63]:
#Though I did not want for this to happen, all the nodes happened to be each individual article......
tree = pi_client.get_tree(doc_id, node_summary=True)

print("Document Tree:")
utils.print_tree(tree)


Document Tree:
{'doc_id': 'pi-cmpwnhsox02sm01qu3njsgetz',
 'status': 'completed',
 'retrieval_ready': True,
 'result': [{'title': 'enriched uranium must remain inside iran...',
             'node_id': '0000',
             'summary': 'Amidst a fragile ceasefire and stalled p...'},
            {'title': 'i vadasseri damodara menon satheesan why...',
             'node_id': '0001',
             'summary': 'Kerala Chief Minister VD Satheesan is fa...'},
            {'title': 'pm modi meloni chemistry how italy becam...',
             'node_id': '0002',
             'summary': 'Facing a volatile global landscape, Indi...'},
            {'title': 'commercial lpg gets costlier cylinder pr...',
             'node_id': '0003',
             'summary': 'Effective June 1, India has increased pr...'},
            {'title': 'dks sidda head to delhi to finalise new ...',
             'node_id': '0004',
             'summary': 'Ahead of the upcoming Karnataka governme...'},
            {'title': 'days

In [12]:
def flatten_tree(nodes):

    for node in nodes:
        for child in node.get("children", []):

            nodes.extend(flatten_tree(child))
    return nodes

In [ ]:
print(flatten_tree(tree))

[{'doc_id': 'pi-cmpw2m54l024o01qurgrp3mq9', 'status': 'completed', 'retrieval_ready': True, 'result': [{'title': 'enriched uranium must remain inside iran mojtaba khamenei rejects key us demand amid fragile ceasefire talks', 'node_id': '0000', 'page_index': 1, 'summary': "Amidst a fragile ceasefire and stalled peace negotiations, Iran's Supreme Leader has rejected a U.S. demand to export its near-weapons-grade uranium stockpile, citing national security concerns. The conflict, characterized by mutual distrust, U.S. blockades, and threats of renewed military strikes, remains deadlocked as Iran insists on security guarantees before discussing its nuclear program. While the U.S. and Israel demand the removal of the stockpile to end the war, Iranian officials have suggested potential compromises, such as diluting the material under international supervision, as the IAEA continues to monitor the status of Iran's enriched uranium reserves.", 'text': '# enriched uranium must remain inside ira

Pure Page Index

In [13]:
async def find_relevant_nodes(tree: dict, query: str) -> list:
    #Here we call the llm to view our tree structure to see which nodes are relevant

    tree_without_text = utils.remove_fields(
        tree.copy(), fields=["text"]
    )

    search_prompt = f"""
    You are a document retrieval expert. Given a question and
    a hierarchical tree structure of a document, identify all
    nodes likely to contain the answer.
    Each node has a node_id, title, and summary.
    Follow cross-references if a section mentions another.
    Question: {query}
    Document tree structure:
    {json.dumps(tree_without_text, indent=2)}
    Reply in this JSON format only:
    {{
        "thinking": "<reasoning about which nodes are relevant>",
        "node_list": ["node_id_1", "node_id_2"]
    }}
    """


    response = await gemini_client.aio.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=search_prompt,
    config={
        "temperature": 0,
        "response_mime_type": "application/json"
        }
      )
    result = json.loads(response.text)
    print(f"LLM reasoning: {result}")
    return result["node_list"]

In [16]:
# This extracts the content from nodes deeemed relevant
def collect_node_content(tree: dict, node_ids: list) -> str:

    all_nodes = flatten_tree(tree["result"])
    context_parts = []
    for node in all_nodes:
        if node["node_id"] in node_ids:
            title = node.get("title", "Untitled")
            pages = f"pages {node.get('start_index', '?')}-{node.get('end_index', '?')}"
            text = node.get("text", "")
            context_parts.append(
                f"[{title} | {pages}]\n{text}"
            )
    return "\n\n---\n\n".join(context_parts)

# This is the LLm call to provide relevant nodes and get final response
async def answer_query(tree: dict, query: str) -> dict:

    #First get relevant node_ids
    node_ids=await find_relevant_nodes(tree, query)
    if not node_ids:
        return {
            "answer": "No relevant information found in the document.",
            "retrieved_nodes": [],
            "context_length": 0,
        }

    #Get the context from the nodes
    context = collect_node_content(tree, node_ids)

    #Fianlly provide it to the llm to get the final answer
    answer_prompt = f"""
    Answer the questions strictly using only the provided context.
    You are not allowed to use any other sources.
    If no context is found, mention so, and don't give any answer.
    Context:
    {context}
    Question: {query}
    """
    response = await gemini_client.aio.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=answer_prompt,
    config={"temperature": 0})

    return {
        "answer": response.text,
        "retrieved_nodes": node_ids,
        "context_length": len(context),
      }


# query = "Explain the relation between India and other foreign countries."
query = "What is the general age of Karnataka's new and old cabinet?"
#Time for response is measured so that further comparisions can be made
tic = time.perf_counter()
result = await (answer_query(tree, query))
toc = time.perf_counter()
print(f"Time taken: {toc - tic:0.4f} seconds")
print(result["answer"])
print(f"Nodes used: {result['retrieved_nodes']}")

LLM reasoning: {'thinking': "The question asks for the general age of Karnataka's new and old cabinet. Node 0014 ('cabinet age debate puts veterans on edge') explicitly discusses the debate regarding the age composition of the new Karnataka cabinet and the potential replacement of veteran ministers from the previous Siddaramaiah administration. Node 0004 provides context on the formation of the new team but does not discuss age, making 0014 the primary source.", 'node_list': ['0014']}
Time taken: 1.7909 seconds
Based on the provided context:

*   **Siddaramaiah's cabinet (old):** The average age of ministers was 64-65 years. Of the thirty-four ministers, twenty-one were above sixty years, including nine in the 70-80 age group.
*   **New cabinet:** The text does not provide the age of a "new" cabinet, as it describes a debate regarding the potential formation of a cabinet by chief minister-designate DK Shivakumar. It notes a push to form a "relatively younger cabinet with those in the 4

Page Index + BM25

In [17]:
!pip install rank-bm25
from rank_bm25 import BM25Okapi
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from typing import List

In [ ]:
print(tree["result"])

[{'title': 'enriched uranium must remain inside iran mojtaba khamenei rejects key us demand amid fragile ceasefire talks', 'node_id': '0000', 'page_index': 1, 'summary': "Amidst fragile ceasefire talks following recent military conflicts, Iran's Supreme Leader has rejected a U.S. demand to export its near-weapons-grade uranium stockpile, citing national security concerns. The standoff persists as both sides remain deeply suspicious of each other's intentions, with the U.S. threatening further strikes and Iran demanding security guarantees. While negotiations are deadlocked over nuclear and regional security issues, potential compromises—such as diluting the stockpile under international supervision—have been proposed to address the status of Iran's remaining nuclear material.", 'text': '# enriched uranium must remain inside iran mojtaba khamenei rejects key us demand amid fragile ceasefire talks\n\niran\'s supreme leader has ordered that the country\'s stockpile of near-weapons-grade e

In [18]:

all_nodes = flatten_tree(tree["result"])
documents = []
#To implement bm25 we first take all the nodes of our tree and store it as documents in a list
for node in all_nodes:
    text = node.get("text", "")
    if text.strip():
        documents.append(Document( page_content=text,
                                  metadata={
                                      "node_id": node.get("node_id"),
                                      "title": node.get("title"),
                                      "summary": node.get("summary")
                                      }
            ))
#Now we take the text from the document and treat each word as token and collect it while maintaining document distinction
tokenized_docs = [doc.page_content.split() for doc in documents]
# We take this tokenized list and input it into the bm25 module so that it can rank the importance of words
bm25 = BM25Okapi(tokenized_docs)

In [46]:
print(tokenized_docs)

[['#', 'enriched', 'uranium', 'must', 'remain', 'inside', 'iran', 'mojtaba', 'khamenei', 'rejects', 'key', 'us', 'demand', 'amid', 'fragile', 'ceasefire', 'talks', "iran's", 'supreme', 'leader', 'has', 'ordered', 'that', 'the', "country's", 'stockpile', 'of', 'near-weapons-grade', 'enriched', 'uranium', 'must', 'remain', 'inside', 'iran,', 'two', 'senior', 'iranian', 'sources', 'told', 'reuters,', 'a', 'move', 'that', 'could', 'further', 'complicate', 'peace', 'negotiations', 'with', 'the', 'united', 'states', 'and', 'israel', 'amid', 'the', 'ongoing', 'regional', 'conflict.', 'according', 'to', 'reuters,', 'ayatollah', 'mojtaba', "khamenei's", 'directive', 'hardens', "tehrain's", 'position', 'on', 'one', 'of', "washington's", 'key', 'demands', 'during', 'negotiations', 'aimed', 'at', 'ending', 'the', 'us-israeli', 'war', 'on', 'iran.', 'israeli', 'officials', 'have', 'told', 'reuters', 'that', 'us', 'president', 'donald', 'trump', 'had', 'assured', 'israel', 'that', 'any', 'peace', 'a

In [47]:
#Here we create  a retriever class and give it the bm25 instance we have already created along with the list of documents
# the idea is to match the query with each of the documents and provide as context to the llm the ones that best match
class BM25PageIndexRetriever(BaseRetriever):
    bm25: BM25Okapi
    documents: List[Document]
    k: int = 4
    def _get_relevant_documents(self,query: str) -> List[Document]:


        query = num2words(query)
        query = re.sub(r'\s+', ' ', query)
        query = query.lower()
        tokenized_query = query.split()
        scores = self.bm25.get_scores(tokenized_query)

        top_indices = sorted(range(len(scores)),key=lambda i: scores[i],reverse=True)[:self.k]
        # Above line sorts the list of whole numbers based on the corresponding value of scores[number] and then takes the first k element
        #In other word take the indices of the top k scores

        return [self.documents[i] for i in top_indices]

In [55]:
retriever = BM25PageIndexRetriever(
    bm25=bm25,
    documents=documents,
    k=2
)

In [62]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# Finally calling llm to look at relevant nodes and answer

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    google_api_key=GOOGLE_API_KEY
)


prompt = ChatPromptTemplate.from_template("""
Answer the questions strictly using only the provided context.
You are not allowed to use any other sources.
If no context is found, mention so, and don't give any answer.
Context:
{context}
Question:
{question}
""")


rag_chain = (
    {   "context": retriever,
        "question": RunnablePassthrough()} | prompt| llm| StrOutputParser())

query="What is the general age of Karnataka's new and old cabinet?"
tic = time.perf_counter()
response = rag_chain.invoke(f"{query}")
toc = time.perf_counter()
print(f"Time taken: {toc - tic:0.4f} seconds")
print(response)

Time taken: 0.7074 seconds
Based on the provided document, the average age of ministers in the previous Siddaramaiah administration was 64-65 years. The document does not provide the age of the "new" cabinet, as it describes a debate regarding its composition and a push to form a relatively younger cabinet.


Naive Model


In [61]:
async def answer_query_naive(query):
    answer_prompt = f"""
    Answer the questions strictly using only the provided context.
    You are not allowed to use any other sources.
    If no relevant information in the context is found, mention so, and don't give any answer.
    Context:
    {complete_context}
    Question: {query}
    """
    response = await gemini_client.aio.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=answer_prompt,
    config={"temperature": 0})

    return {
        "answer": response.text}

query = "What is the general age of Karnataka's new and old cabinet?"
tic = time.perf_counter()
result = await (answer_query_naive(query))
toc = time.perf_counter()
print(f"Time taken: {toc - tic:0.4f} seconds")
print(result["answer"])

Time taken: 0.9909 seconds
Based on the provided context, the average age of the ministers in the cabinet under former Chief Minister Siddaramaiah was 64–65 years. Regarding the new cabinet, the text mentions that the Congress high command is advocating for a "kerala-style model" that blends experience with younger legislators, with a strong push to form a cabinet with those in the 40–60 age group.


In [ ]:
#We see that our model(Page index + bm25) is faster then the naive version but the pure page index takes double
# the time as these two models, we see that llm calling is expensive and takes siginificant-majority of the time,
# hence it was outperformed at double speed by both model 2 and 3
# But PageIndex has its own benefits regardless in being able to organize data in a sementcially structured way,
# So that the BM model could retrieve it and increse the efficiency and reducing latency,
# causing model 2 to outperform model 3